# 00. Pilot subset and Earth Engine downloads

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.pilots import climate, clustering, data, earth_engine, evaluation, features
from experiments.tuning import tuning
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
AREA = "pilots"


def show_and_save(figure, run, name):
    figure.tight_layout()
    figure.savefig(run / f"{name}.png", dpi=150)
    display(figure)
    plt.close(figure)


def repeated(name, X, rows, ok, seeds=(1, 2, 3), params=evaluation.PARAMS):
    """Mean and SD over random-forest seeds of every evaluation view."""
    summaries, details = [], []
    for seed in seeds:
        summary, holdout = evaluation.evaluate(name, X, rows, ok, params, seed)
        summaries.append({**summary, "seed": seed})
        details.extend({**r, "seed": seed} for r in holdout)
    return summaries, details


def summarize(summaries):
    table = pd.DataFrame(summaries)
    metrics = ["random_split_macro_f1", "spatial_cv_macro_f1", "dataset_holdout_macro_f1", "jecam_macro_f1",
               "jecam_carrot_f1", "jecam_cabbage_f1", "jecam_potato_f1"]
    grouped = table.groupby("feature_set", sort=False)
    out = grouped[metrics].agg(["mean", "std"]).round(3)
    out.insert(0, ("pixels_used", ""), grouped["pixels_used"].first())
    return out

## Inputs

In [ ]:
quality_path = ""  # Optional quality.json from Preprocessing 01; blank uses its verified pointer
per_cell = {"train": 40, "validation": 15, "test": None}  # parcels per (partition, crop, source); None keeps all JECAM
pixels_per_parcel = 10
settings = json.loads((root / "settings.local.json").read_text())
quality_handoff, quality = data.load_quality(root, quality_path or None)
print("Quality handoff:", quality_handoff)

## Sample parcels and read annual Sentinel-2 series

In [ ]:
run = tuning.new_run(root, "subset", quality_handoff, area=AREA, per_cell={k: str(v) for k, v in per_cell.items()},
                     pixels_per_parcel=pixels_per_parcel)
regions = data.sample_parcels(quality["regions"], per_cell)
rows, s2_values, s2_valid = data.load_series(regions, quality["pixel_paths"], pixels_per_parcel)
print("Fragments:", len(regions), "| pixels:", len(rows))
display(pd.DataFrame(rows).groupby(["partition", "crop", "source_dataset"]).partition_group.nunique().rename("parcels").reset_index())

## Sentinel-1 half-month backscatter

In [ ]:
ee = earth_engine.initialize(settings["earth_engine_project"])
s1_values = earth_engine.sample_s1(ee, rows, run / "cache", progress=lambda m: print(m, flush=True))
print("Pixels with at least 20 half-months of Sentinel-1:", int((np.isfinite(s1_values[..., 0]).sum(axis=1) >= 20).sum()))

## ERA5-Land monthly temperature per parcel

In [ ]:
parcel_points = {}
for row in rows:
    parcel_points.setdefault(row["partition_group"], {"key": row["partition_group"], "lon": row["lon"], "lat": row["lat"], "year": row["year"]})
era5 = earth_engine.era5_monthly(ee, list(parcel_points.values()), run / "cache")
print("Parcels with temperature:", sum(v is not None and None not in v for v in era5.values()), "of", len(parcel_points))

## Paoay inputs

In [ ]:
from crop_pipeline.paoay import aligned
def local_path(value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (root / path).resolve()
kmz_root = local_path(settings.get("paoay_kmz_root", "./data/paoay"))
kmz_paths = [kmz_root / name for name in ("00 - Region of Interest.kmz", "00 - Atok Boundary.kmz", "Crop Area.kmz")]
imagery_root = local_path(settings["atok_imagery_root"])
import rasterio
with rasterio.open(next(iter(sorted(imagery_root.glob("*/Sentinel2_CSPlus_*.tif"))))) as source:
    crs = source.crs.to_string()
roi, areas, _ = aligned.study_area(kmz_paths, settings.get("paoay_roi_name", "Region of Interest"),
                                   list(settings.get("paoay_context_names", [])), crs)
grid, mask = aligned.grid_for(imagery_root, roi)
year = aligned.choose_year(aligned.year_completeness(imagery_root, list(range(2019, 2026)), grid, mask))
paoay_cube, _ = aligned.read_year(imagery_root, year, grid)
paoay_s1 = earth_engine.s1_grid(ee, year, grid, run / "cache")
lon, lat = roi.representative_point().coords[0]
from pyproj import Transformer
lon, lat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(lon, lat)
paoay_era5 = earth_engine.era5_monthly(ee, [{"key": "paoay", "lon": lon, "lat": lat, "year": year}], run / "cache")["paoay"]
print("Paoay year:", year, "| study pixels:", int(mask.sum()), "| grid:", mask.shape)

## Save

In [ ]:
data.save(run, rows, {"s2_values": s2_values, "s2_valid": s2_valid, "s1_values": s1_values,
                      "paoay_cube": paoay_cube, "paoay_s1": paoay_s1, "paoay_mask": mask})
(run / "era5.json").write_text(json.dumps(era5))
(run / "paoay.json").write_text(json.dumps({"year": year, "grid": grid, "era5": paoay_era5, "lon": lon, "lat": lat,
                                            "kmz_paths": [str(p) for p in kmz_paths]}))
print("Run folder:", run)